# 🧠 Knowledge Structure & LLM Reasoning Quality
## Real LLM Experiment — Claude Sonnet 4.6 + GPT-4o

---

**Article:** *From Data to Knowledge: Why Knowledge Graphs and Network Science Are a Likely Foundation of More Reliable AI*

This notebook runs the full experiment and automatically generates two publication-ready figures:

| Output file | What it contains |
|---|---|
| `experiment_llm_results.png` | 5-panel figure: performance, hallucinations, topology, delta |
| `results_table.png` | Styled results table for Substack / Medium |

Both files are downloaded automatically at the end.

---

### ⚠️ Declared limitations
- Knowledge base is **synthetic** (34 triples, narrow biomedical domain)
- Scoring is **keyword-based** — a heuristic, not a validated NLP metric
- Sample is small: 4 questions × 2 conditions × 2 models
- Results are **preliminary exploratory evidence**, not statistically validated claims

---

**GitHub:** `github.com/YOUR_USERNAME/knowledge-graph-llm-experiment`

## ⚙️ Section 1 — Install Dependencies

In [ ]:
!pip install -q anthropic openai networkx matplotlib numpy
print('✓ Dependencies ready')

## 🔑 Section 2 — API Keys

**Recommended:** Use Colab Secrets (🔑 icon in the left sidebar).  
Add secrets named `ANTHROPIC_API_KEY` and/or `OPENAI_API_KEY` — the cell reads them automatically without exposing your keys in the notebook.

You only need **one** key. The experiment auto-detects which models are available.

In [ ]:
import os

# ── Option A: Colab Secrets (recommended) ──────────────────────────────────
try:
    from google.colab import userdata
    if userdata.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    if userdata.get('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('✓ Keys loaded from Colab Secrets')
except Exception:
    print('ℹ Colab Secrets not available — set keys manually below')

# ── Option B: paste key manually (remove # before the line) ────────────────
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# os.environ['OPENAI_API_KEY']    = 'sk-...'

# ── Validate ────────────────────────────────────────────────────────────────
has_anthropic = bool(os.environ.get('ANTHROPIC_API_KEY'))
has_openai    = bool(os.environ.get('OPENAI_API_KEY'))
print(f'  Anthropic key: {"✓ found" if has_anthropic else "✗ not set"}')
print(f'  OpenAI key:    {"✓ found" if has_openai    else "✗ not set"}')
if not has_anthropic and not has_openai:
    raise EnvironmentError('Please set at least one API key above.')

## 📚 Section 3 — Imports & API Clients

In [ ]:
import os, json, time, random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from networkx.algorithms.community import greedy_modularity_communities
from datetime import datetime

random.seed(42)

MODELS = {}

if os.environ.get('ANTHROPIC_API_KEY'):
    try:
        import anthropic
        anthropic_client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
        MODELS['claude-sonnet-4-6'] = 'Anthropic'
        print('✓ Anthropic client ready  →  claude-sonnet-4-6')
    except Exception as e:
        print(f'✗ Anthropic error: {e}')

if os.environ.get('OPENAI_API_KEY'):
    try:
        import openai
        openai_client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        MODELS['gpt-4o'] = 'OpenAI'
        print('✓ OpenAI client ready  →  gpt-4o')
    except Exception as e:
        print(f'✗ OpenAI error: {e}')

print(f'\nModels active: {", ".join(MODELS.keys())}')

## 🗄️ Section 4 — Synthetic Knowledge Base

34 triples across 21 biomedical entities covering drug → mechanism → disease relationships.

In [ ]:
TRIPLES = [
    ('Aspirin',           'inhibits',             'COX1'),
    ('Aspirin',           'inhibits',             'COX2'),
    ('Ibuprofen',         'inhibits',             'COX1'),
    ('Ibuprofen',         'inhibits',             'COX2'),
    ('Metformin',         'activates',            'AMPK'),
    ('Metformin',         'inhibits',             'mTOR'),
    ('Atorvastatin',      'inhibits',             'HMG-CoA_Reductase'),
    ('Celecoxib',         'selectively_inhibits',  'COX2'),
    ('COX1',              'produces',             'Prostaglandins'),
    ('COX2',              'produces',             'Prostaglandins'),
    ('COX2',              'mediates',             'Inflammation'),
    ('Prostaglandins',    'cause',                'Inflammation'),
    ('Prostaglandins',    'cause',                'Pain'),
    ('AMPK',              'reduces',              'Glucose_Production'),
    ('mTOR',              'promotes',             'Cell_Growth'),
    ('HMG-CoA_Reductase', 'synthesizes',          'Cholesterol'),
    ('Aspirin',           'treats',               'Inflammation'),
    ('Aspirin',           'treats',               'Cardiovascular_Disease'),
    ('Ibuprofen',         'treats',               'Inflammation'),
    ('Ibuprofen',         'treats',               'Pain'),
    ('Metformin',         'treats',               'Type2_Diabetes'),
    ('Atorvastatin',      'treats',               'Cardiovascular_Disease'),
    ('Celecoxib',         'treats',               'Inflammation'),
    ('Cardiovascular_Disease', 'involves',        'Inflammation'),
    ('Type2_Diabetes',    'involves',             'AMPK'),
    ('Type2_Diabetes',    'involves',             'mTOR'),
    ('Cardiovascular_Disease', 'associated_with', 'Cholesterol'),
    ('Aspirin',           'inhibits',             'Platelet_Aggregation'),
    ('Ibuprofen',         'increases_risk',       'Gastric_Ulcer'),
    ('COX1',              'protects',             'Gastric_Mucosa'),
    ('Celecoxib',         'spares',               'COX1'),
    ('Celecoxib',         'reduces_risk',         'Gastric_Ulcer'),
    ('Atorvastatin',      'reduces',              'Inflammation'),
    ('Cholesterol',       'promotes',             'Cardiovascular_Disease'),
]

ENTITIES = sorted(set(s for s,_,_ in TRIPLES) | set(o for _,_,o in TRIPLES))
print(f'Knowledge Base: {len(TRIPLES)} triples, {len(ENTITIES)} entities')

## 🔬 Section 5 — Network Analysis

Degree centrality, betweenness centrality, community detection (Greedy Modularity), and structural robustness.

In [ ]:
G  = nx.DiGraph()
for s, r, o in TRIPLES:
    G.add_edge(s, o, relation=r)
ug = G.to_undirected()

degree_c  = nx.degree_centrality(ug)
between_c = nx.betweenness_centrality(ug)
try:
    eigen_c = nx.eigenvector_centrality(ug, max_iter=500)
except:
    eigen_c = {n: 0.0 for n in ug.nodes()}

communities = list(greedy_modularity_communities(ug))
node_comm   = {n: i for i, comm in enumerate(communities) for n in comm}
top_hub     = max(degree_c, key=degree_c.get)
G_pruned    = ug.copy()
G_pruned.remove_node(top_hub)
robustness  = len(max(nx.connected_components(G_pruned), key=len)) / (len(ug) - 1)
top_hubs    = sorted(degree_c.items(),  key=lambda x: -x[1])[:5]
top_bridges = sorted(between_c.items(), key=lambda x: -x[1])[:5]

metrics = dict(
    n_nodes=G.number_of_nodes(), n_edges=G.number_of_edges(),
    density=nx.density(ug), avg_clustering=nx.average_clustering(ug),
    n_communities=len(communities), top_hub=top_hub,
    robustness=robustness, top_hubs=top_hubs, top_bridges=top_bridges,
    node_comm=node_comm, degree_c=degree_c, between_c=between_c
)

print(f'Nodes: {metrics["n_nodes"]}  |  Edges: {metrics["n_edges"]}')
print(f'Density: {metrics["density"]:.3f}  |  Avg Clustering: {metrics["avg_clustering"]:.3f}')
print(f'Communities: {metrics["n_communities"]}  |  Top hub: {metrics["top_hub"]}')
print(f'Structural robustness: {metrics["robustness"]:.1%}')
print(f'\nTop hubs (degree centrality):')
for n, s in top_hubs: print(f'  {n:<30} {s:.3f}')
print(f'\nTop bridges (betweenness centrality):')
for n, s in top_bridges: print(f'  {n:<30} {s:.3f}')

## 📝 Section 6 — Contexts & Questions

In [ ]:
def build_plain_text(triples):
    sentences = [
        f"{s.replace('_',' ')} {r.replace('_',' ')} {o.replace('_',' ')}."
        for s, r, o in triples
    ]
    random.shuffle(sentences)
    return 'The following is a collection of biomedical facts:\n\n' + ' '.join(sentences)

def build_kg_context(triples, metrics):
    lines = ['=== BIOMEDICAL KNOWLEDGE GRAPH ===\n', '-- Relationships --']
    for s, r, o in triples:
        lines.append(f'  ({s}) --[{r}]--> ({o})')
    lines.append('\n-- Network Analysis --')
    lines.append(f'  Entities: {metrics["n_nodes"]} | Relationships: {metrics["n_edges"]}')
    lines.append(f'  Density: {metrics["density"]:.3f} | Communities: {metrics["n_communities"]}')
    lines.append('\n-- Top Structural Hubs (degree centrality) --')
    for node, score in metrics['top_hubs']:
        lines.append(f'  {node}: {score:.3f}')
    lines.append('\n-- Top Knowledge Bridges (betweenness centrality) --')
    for node, score in metrics['top_bridges']:
        lines.append(f'  {node}: {score:.3f}')
    lines.append(f'\n-- Structural Robustness --')
    lines.append(
        f'  After removing top hub ({metrics["top_hub"]}): '
        f'{metrics["robustness"]:.1%} of graph remains connected'
    )
    return '\n'.join(lines)

PLAIN_TEXT = build_plain_text(TRIPLES)
KG_CONTEXT = build_kg_context(TRIPLES, metrics)
print(f'Plain Text context: {len(PLAIN_TEXT)} chars')
print(f'KG context:         {len(KG_CONTEXT)} chars')

In [ ]:
QUESTIONS = [
    {
        'id': 'Q1', 'type': 'Multi-hop Reasoning', 'n_hops': 4,
        'question': (
            'Why might Celecoxib cause fewer gastric side effects than Ibuprofen, '
            'even though both treat inflammation? Explain the mechanism step by step.'
        ),
        'correct_keywords':     ['COX1', 'COX2', 'selective', 'gastric', 'mucosa', 'spares'],
        'hallucination_traps': ['kidney failure', 'serotonin', 'liver toxicity',
                                 'blood pressure', 'dopamine', 'histamine'],
        'ground_truth': (
            'Celecoxib selectively inhibits COX2 and spares COX1. '
            'COX1 protects the gastric mucosa. '
            'Ibuprofen inhibits both COX1 and COX2, removing gastric protection.'
        ),
    },
    {
        'id': 'Q2', 'type': 'Cross-domain Bridge', 'n_hops': 2,
        'question': (
            'Through what mechanism could Atorvastatin potentially benefit patients '
            'with Cardiovascular Disease beyond its cholesterol-lowering effect? '
            'Use only information provided.'
        ),
        'correct_keywords':     ['inflammation', 'anti-inflammatory', 'cardiovascular',
                                  'reduces', 'Atorvastatin'],
        'hallucination_traps': ['blood sugar', 'insulin resistance', 'kidney',
                                 'autoimmune', 'platelet', 'serotonin'],
        'ground_truth': (
            'Atorvastatin reduces Inflammation. '
            'Cardiovascular Disease involves Inflammation. '
            'Therefore Atorvastatin may benefit cardiovascular disease via anti-inflammatory effect.'
        ),
    },
    {
        'id': 'Q3', 'type': 'Structural Hub', 'n_hops': 5,
        'question': (
            'Which biological process or molecule connects the greatest number of drugs '
            'in this knowledge base? Name it and explain what that structural centrality '
            'implies about its therapeutic importance.'
        ),
        'correct_keywords':     ['inflammation', 'multiple', 'central', 'hub',
                                  'connected', 'drugs', 'COX'],
        'hallucination_traps': ['DNA repair', 'apoptosis', 'protein folding',
                                 'autophagy', 'mitosis', 'immune checkpoint'],
        'ground_truth': (
            'Inflammation is the central hub, connected to Aspirin, Ibuprofen, '
            'Celecoxib, Atorvastatin via COX2/Prostaglandins pathway.'
        ),
    },
    {
        'id': 'Q4', 'type': 'Multi-domain Mechanism', 'n_hops': 5,
        'question': (
            'A patient has both Type 2 Diabetes and Cardiovascular Disease. '
            'Based solely on the information provided, what shared molecular '
            'mechanism links these two conditions?'
        ),
        'correct_keywords':     ['inflammation', 'AMPK', 'mTOR', 'shared', 'mechanism',
                                  'molecular', 'involves'],
        'hallucination_traps': ['genetic mutation', 'viral infection', 'autoimmune',
                                 'obesity gene', 'microbiome', 'cortisol'],
        'ground_truth': (
            'Type 2 Diabetes involves AMPK and mTOR. '
            'Cardiovascular Disease involves Inflammation. '
            'Both share downstream inflammatory pathways via Prostaglandins/COX2.'
        ),
    },
]
print(f'{len(QUESTIONS)} questions defined')
for q in QUESTIONS:
    print(f'  [{q["id"]}] {q["type"]}  ({q["n_hops"]} hops)')

## 🤖 Section 7 — API Call & Scoring Functions

In [ ]:
SYSTEM_PLAIN = """You are a biomedical reasoning assistant.
Answer questions based ONLY on the information provided in the text below.
Do not use any external knowledge or training data beyond what is explicitly stated.
If the answer cannot be found in the provided information, say exactly: "Not found in provided information."
Be precise and complete in your reasoning."""

SYSTEM_KG = """You are a biomedical reasoning assistant working with a structured Knowledge Graph.
Answer questions based ONLY on the relationships explicitly defined in the Knowledge Graph below.
You may traverse multiple relationships (multi-hop reasoning) to reach your answer.
Do not use any external knowledge beyond what is explicitly stated in the graph.
Show your reasoning path through the graph when relevant.
If the answer cannot be derived from the graph, say exactly: "Not derivable from graph."""


def call_claude(system_prompt, context, question, model='claude-sonnet-4-6'):
    full_user = f'{context}\n\n---\nQuestion: {question}'
    response  = anthropic_client.messages.create(
        model=model, max_tokens=800, system=system_prompt,
        messages=[{'role': 'user', 'content': full_user}]
    )
    return response.content[0].text


def call_openai(system_prompt, context, question, model='gpt-4o'):
    full_user = f'{context}\n\n---\nQuestion: {question}'
    response  = openai_client.chat.completions.create(
        model=model, max_tokens=800,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': full_user}
        ]
    )
    return response.choices[0].message.content


def call_model(model_id, provider, system_prompt, context, question):
    if provider == 'Anthropic':
        return call_claude(system_prompt, context, question, model=model_id)
    return call_openai(system_prompt, context, question, model=model_id)


def score_response(response, question):
    rl   = response.lower()
    hits = [kw for kw in question['correct_keywords'] if kw.lower() in rl]
    hals = [t  for t  in question['hallucination_traps'] if t.lower() in rl]
    refused = any(p in rl for p in [
        'not found in provided', 'not derivable from graph',
        'cannot be found', 'not mentioned', 'no information'
    ])
    words = len(response.split())
    return {
        'keyword_recall':      round(len(hits) / len(question['correct_keywords']), 3),
        'keywords_found':      hits,
        'hallucinations':      hals,
        'hallucination_count': len(hals),
        'refused':             refused,
        'reasoning_depth':     round(min(words / 100, 1.0), 3),
        'path_coherence':      round(len(hits) / max(len(question['correct_keywords']), 1), 3),
        'response_words':      words,
    }

print('Functions defined ✓')

## 🚀 Section 8 — Run the Experiment

> ⏱️ **~2–4 min** depending on API latency  
> Total calls: `2 models × 4 questions × 2 conditions = 16 API calls`

In [ ]:
print('=' * 65)
print('  EXPERIMENT: Knowledge Structure and LLM Reasoning Quality')
print('=' * 65)
print(f'  Domain   : Synthetic Biomedical (Drug-Disease-Mechanism)')
print(f'  KB size  : {len(TRIPLES)} triples, {len(ENTITIES)} entities')
print(f'  Questions: {len(QUESTIONS)}')
print(f'  Models   : {", ".join(MODELS.keys())}')
print(f'  Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('=' * 65)

all_results = {}

for model_id, provider in MODELS.items():
    print(f'\n{"─"*65}')
    print(f'  MODEL: {model_id}  ({provider})')
    print(f'{"─"*65}')
    model_results = []

    for q in QUESTIONS:
        print(f"\n  [{q['id']}] {q['type']} ({q['n_hops']} hops)")

        print('    → [plain] calling API...', end=' ', flush=True)
        resp_plain = call_model(model_id, provider, SYSTEM_PLAIN,
                                f'INFORMATION:\n{PLAIN_TEXT}', q['question'])
        time.sleep(1)
        print('done')

        print('    → [kg]    calling API...', end=' ', flush=True)
        resp_kg = call_model(model_id, provider, SYSTEM_KG, KG_CONTEXT, q['question'])
        time.sleep(1)
        print('done')

        sc_plain = score_response(resp_plain, q)
        sc_kg    = score_response(resp_kg,    q)

        model_results.append({
            'question_id': q['id'], 'type': q['type'], 'n_hops': q['n_hops'],
            'plain': {'response': resp_plain, 'scores': sc_plain},
            'kg':    {'response': resp_kg,    'scores': sc_kg},
        })

        print(f"    Plain  kw:{sc_plain['keyword_recall']:.0%}  "
              f"hal:{sc_plain['hallucination_count']}  depth:{sc_plain['reasoning_depth']:.2f}  "
              f"refused:{sc_plain['refused']}")
        print(f"    KG     kw:{sc_kg['keyword_recall']:.0%}  "
              f"hal:{sc_kg['hallucination_count']}  depth:{sc_kg['reasoning_depth']:.2f}  "
              f"refused:{sc_kg['refused']}")

    all_results[model_id] = model_results

print('\n✓ All API calls complete')

## 📊 Section 9 — Aggregate Results

In [ ]:
def avg(lst): return round(sum(lst) / len(lst), 3)

summary    = {}
model_ids  = list(MODELS.keys())

print('=' * 65)
print(f"  {'Model':<28} {'Cond':<8} {'KW Recall':>10} {'Halluc':>8} {'Depth':>8}")
print(f"  {'-'*62}")

for model_id, results in all_results.items():
    for cond in ['plain', 'kg']:
        kw  = avg([r[cond]['scores']['keyword_recall']      for r in results])
        hal = avg([r[cond]['scores']['hallucination_count'] for r in results])
        dep = avg([r[cond]['scores']['reasoning_depth']     for r in results])
        summary.setdefault(model_id, {})[cond] = {
            'keyword_recall': kw, 'hallucinations': hal, 'reasoning_depth': dep
        }
        print(f'  {model_id:<28} {cond:<8} {kw:>10.1%} {hal:>8.2f} {dep:>8.2f}')

print(f'\n  Network: {metrics["n_nodes"]} nodes, {metrics["n_edges"]} edges')
print(f'  Top hub: {metrics["top_hub"]}  |  Communities: {metrics["n_communities"]}')
print(f'  Structural robustness: {metrics["robustness"]:.1%}')
print('=' * 65)

## 📈 Section 10 — Figure A: 5-Panel Experiment Results

Generates **`experiment_llm_results.png`** — the main figure for the article.

In [ ]:
ACCENT = '#1A5276'; GRAY_C = '#AEB6BF'; BLUE2 = '#2E86C1'
RED_C  = '#C0392B'; GREEN  = '#117A65'
BG     = '#F8FBFF'
MODEL_COLORS = {
    model_ids[0]: ACCENT,
    model_ids[-1]: RED_C,
}
n_models = len(model_ids)

fig = plt.figure(figsize=(20, 14), facecolor=BG)
gs  = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# ── A: Keyword Recall ─────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(BG)
x, w = np.arange(n_models), 0.32
plain_kw = [summary[m]['plain']['keyword_recall'] for m in model_ids]
kg_kw    = [summary[m]['kg']['keyword_recall']    for m in model_ids]
ax1.bar(x - w/2, plain_kw, w, label='Plain Text',      color=GRAY_C, edgecolor='white', lw=1.5)
ax1.bar(x + w/2, kg_kw,    w, label='Knowledge Graph', color=ACCENT, edgecolor='white', lw=1.5)
ax1.set_ylim(0, 1.25); ax1.set_xticks(x)
ax1.set_xticklabels([m.replace('-', '\n') for m in model_ids], fontsize=9)
ax1.set_title('A. Keyword Recall\nPlain Text vs Knowledge Graph',
              fontsize=11, fontweight='bold', color=ACCENT, pad=8)
ax1.legend(fontsize=8)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
for bar in ax1.patches:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

# ── B: Hallucinations ─────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(BG)
plain_hal = [summary[m]['plain']['hallucinations'] for m in model_ids]
kg_hal    = [summary[m]['kg']['hallucinations']    for m in model_ids]
ax2.bar(x - w/2, plain_hal, w, color=RED_C,  edgecolor='white', lw=1.5, label='Plain Text')
ax2.bar(x + w/2, kg_hal,    w, color=GREEN,  edgecolor='white', lw=1.5, label='Knowledge Graph')
ax2.set_ylim(0, max(plain_hal + [0.5]) * 1.4)
ax2.set_xticks(x); ax2.set_xticklabels([m.replace('-', '\n') for m in model_ids], fontsize=9)
ax2.set_title('B. Hallucinations per Question\n(lower = better)',
              fontsize=11, fontweight='bold', color=ACCENT, pad=8)
ax2.legend(fontsize=8)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
for bar in ax2.patches:
    if bar.get_height() > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

# ── C: Keyword Recall by question ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(BG)
q_ids      = [r['question_id'] for r in list(all_results.values())[0]]
xq         = np.arange(len(q_ids))
bar_w      = 0.18
c_list     = [GRAY_C, ACCENT, '#E67E22', GREEN]
for mi, model_id in enumerate(model_ids):
    results = all_results[model_id]
    for ci, (cond, col) in enumerate([('plain', c_list[mi*2]), ('kg', c_list[mi*2+1])]):
        vals   = [r[cond]['scores']['keyword_recall'] for r in results]
        offset = (mi * 2 + ci - 1.5) * bar_w
        ax3.bar(xq + offset, vals, bar_w, color=col, edgecolor='white', lw=1,
                label=f'{model_id.split("-")[0]} {cond}')
ax3.set_ylim(0, 1.25); ax3.set_xticks(xq); ax3.set_xticklabels(q_ids)
ax3.set_title('C. Keyword Recall by Question\nAll models & conditions',
              fontsize=11, fontweight='bold', color=ACCENT, pad=8)
ax3.legend(fontsize=7, loc='lower right')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# ── D: KG Topology ───────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0:2])
ax4.set_facecolor(BG)
pos = nx.spring_layout(ug, seed=7, k=2.5)
comm_colors = [ACCENT, RED_C, '#117A65', '#784212', '#6C3483']
nc = [comm_colors[metrics['node_comm'].get(n, 0) % len(comm_colors)] for n in ug.nodes()]
ns = [600 + 3500 * metrics['degree_c'].get(n, 0) for n in ug.nodes()]
nx.draw_networkx_edges(ug, pos, ax=ax4, alpha=0.22, edge_color='#AED6F1', width=1.3)
nx.draw_networkx_nodes(ug, pos, ax=ax4, node_color=nc, node_size=ns, alpha=0.92)
top_label_nodes = {n for n, _ in top_hubs[:6]} | {n for n, _ in top_bridges[:4]}
labels = {n: n.replace('_', '\n') for n in ug.nodes() if n in top_label_nodes}
nx.draw_networkx_labels(ug, pos, labels, ax=ax4,
                        font_size=7, font_color='white', font_weight='bold')
patches = [mpatches.Patch(color=comm_colors[i], label=f'Community {i+1}')
           for i in range(min(len(communities), 4))]
ax4.legend(handles=patches, fontsize=8, loc='lower left')
ax4.set_title('D. Knowledge Graph Topology\n(node size = degree centrality, color = community)',
              fontsize=11, fontweight='bold', color=ACCENT, pad=8)
ax4.axis('off')

# ── E: Delta KG vs Plain ──────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor(BG)
metric_labels = ['KW Recall\n(+better)', 'Hallucinations\n(-better)', 'Depth\n(+better)']
xm    = np.arange(len(metric_labels))
bar_wm = 0.35 / max(n_models, 1)
for i, model_id in enumerate(model_ids):
    s = summary[model_id]
    deltas = [
        s['kg']['keyword_recall']   - s['plain']['keyword_recall'],
        -(s['kg']['hallucinations'] - s['plain']['hallucinations']),
        s['kg']['reasoning_depth']  - s['plain']['reasoning_depth'],
    ]
    offset = (i - (n_models - 1) / 2) * (bar_wm + 0.05)
    color  = list(MODEL_COLORS.values())[i % len(MODEL_COLORS)]
    bars   = ax5.bar(xm + offset, deltas, bar_wm + 0.1,
                     color=color, edgecolor='white', lw=1.5,
                     label=model_id.split('-')[0], alpha=0.9)
    for bar, val in zip(bars, deltas):
        ax5.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.005 if val >= 0 else -0.025),
                 f'{val:+.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax5.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax5.set_xticks(xm); ax5.set_xticklabels(metric_labels, fontsize=9)
ax5.set_title('E. KG Improvement over Plain Text\n(Δ per model)',
              fontsize=11, fontweight='bold', color=ACCENT, pad=8)
ax5.legend(fontsize=9)
ax5.spines['top'].set_visible(False); ax5.spines['right'].set_visible(False)

model_str = ' vs '.join(model_ids)
fig.suptitle(
    f'Knowledge Structure and LLM Reasoning — Real LLM Experiment\n{model_str}',
    fontsize=13, fontweight='bold', color='#1C1C1C', y=1.01
)
plt.tight_layout()
plt.savefig('experiment_llm_results.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Saved: experiment_llm_results.png')

## 📋 Section 11 — Figure B: Results Table

Generates **`results_table.png`** — the styled table for Substack / Medium.  
Values come directly from the experiment results above.

In [ ]:
# ── Compute aggregate values from real results ────────────────────────────────
# Average across all models for the table (or use first model if only one)
plain_kw_avg  = avg([summary[m]['plain']['keyword_recall']  for m in model_ids])
kg_kw_avg     = avg([summary[m]['kg']['keyword_recall']     for m in model_ids])
plain_hal_avg = avg([summary[m]['plain']['hallucinations']  for m in model_ids])
kg_hal_avg    = avg([summary[m]['kg']['hallucinations']     for m in model_ids])
plain_dep_avg = avg([summary[m]['plain']['reasoning_depth'] for m in model_ids])
kg_dep_avg    = avg([summary[m]['kg']['reasoning_depth']    for m in model_ids])

delta_kw  = kg_kw_avg  - plain_kw_avg
delta_hal = kg_hal_avg - plain_hal_avg
delta_dep = kg_dep_avg - plain_dep_avg

print(f'Plain Text  →  KW: {plain_kw_avg:.1%}  Hal: {plain_hal_avg:.2f}  Depth: {plain_dep_avg:.2f}')
print(f'KG          →  KW: {kg_kw_avg:.1%}    Hal: {kg_hal_avg:.2f}  Depth: {kg_dep_avg:.2f}')
print(f'Δ           →  KW: {delta_kw:+.1%}  Hal: {delta_hal:+.2f}  Depth: {delta_dep:+.2f}')

In [ ]:
# ── Table figure ──────────────────────────────────────────────────────────────
BG_TABLE  = '#F8FBFF'
BG_ROW    = '#EAF2FB'
ACCENT    = '#1A5276'
GREEN     = '#117A65'
RED_C     = '#C0392B'

rows = [
    ['Keyword Recall (avg)',
     f'{plain_kw_avg:.1%}',
     f'{kg_kw_avg:.1%}',
     f'{delta_kw:+.1%}'],
    ['Hallucinations per Q (avg)',
     f'{plain_hal_avg:.2f}',
     f'{kg_hal_avg:.2f}',
     f'{delta_hal:+.2f}'],
    ['Reasoning Depth (avg)',
     f'{plain_dep_avg:.2f}',
     f'{kg_dep_avg:.2f}',
     f'{delta_dep:+.2f}'],
]

fig2, ax = plt.subplots(figsize=(11, 4.2), facecolor=BG_TABLE)
ax.set_facecolor(BG_TABLE)
ax.axis('off')

col_headers = ['Metric', 'Plain Text', 'Knowledge Graph', 'Δ']
col_x       = [0.04, 0.40, 0.60, 0.82]
col_align   = ['left', 'center', 'center', 'center']
col_w       = [0.34, 0.185, 0.185, 0.155]
row_y       = [0.62, 0.40, 0.18]
header_y    = 0.835
row_h       = 0.195

# Header background
ax.add_patch(FancyBboxPatch((0.02, 0.745), 0.96, 0.135,
    boxstyle='round,pad=0.005', linewidth=0,
    facecolor=ACCENT, transform=ax.transAxes, zorder=2))

# Header text
for label, x, align, w in zip(col_headers, col_x, col_align, col_w):
    ax.text(x + (w/2 if align == 'center' else 0.005), header_y, label,
            ha=align, va='center', fontsize=11.5, fontweight='bold',
            color='white', transform=ax.transAxes, zorder=3)

# Row backgrounds
for ri, y in enumerate(row_y):
    color = BG_ROW if ri % 2 == 0 else '#FFFFFF'
    ax.add_patch(FancyBboxPatch((0.02, y - row_h/2 + 0.005), 0.96, row_h - 0.01,
        boxstyle='round,pad=0.003', linewidth=0,
        facecolor=color, transform=ax.transAxes, zorder=1))

# Row data
for ri, (row, y) in enumerate(zip(rows, row_y)):
    for ci, (val, x, align, w) in enumerate(zip(row, col_x, col_align, col_w)):
        if ci == 3:   # delta column
            c = GREEN if val.startswith('+') else (RED_C if val.startswith('−') or val.startswith('-') else ACCENT)
            fw = 'bold'
        elif ci == 2: # KG column
            c, fw = ACCENT, 'bold'
        elif ci == 0:
            c, fw = '#1c1c1e', 'normal'
        else:
            c, fw = '#4a4a52', 'normal'
        xpos = x + (w/2 if align == 'center' else 0.008)
        ax.text(xpos, y, val, ha=align, va='center',
                fontsize=11, fontweight=fw, color=c,
                transform=ax.transAxes, zorder=3)

# Separator lines
for y in [0.515, 0.295]:
    ax.plot([0.03, 0.97], [y, y], color='#d4cfc8', linewidth=0.7,
            transform=ax.transAxes, zorder=2)

# Title
model_str_short = ' vs '.join([m.split('-')[0].capitalize() for m in model_ids])
ax.text(0.50, 0.965,
        'Experiment Results — Plain Text vs. Knowledge Graph Context',
        ha='center', va='center', fontsize=12.5, fontweight='bold',
        color=ACCENT, transform=ax.transAxes)
ax.text(0.50, 0.930,
        f'Synthetic Biomedical KB · 4 multi-hop questions · Models: {model_str_short}',
        ha='center', va='center', fontsize=9, color='#6b7280',
        transform=ax.transAxes, style='italic')

# Legend
for xpos, color, label in [
    (0.38, GREEN,  '▲ KG improvement'),
    (0.56, RED_C,  '▼ KG reduction (hallucinations — lower is better)'),
]:
    ax.text(xpos, 0.035, label, ha='left', va='center',
            fontsize=8.5, color=color, transform=ax.transAxes)

# Outer border
ax.add_patch(FancyBboxPatch((0.01, 0.01), 0.98, 0.975,
    boxstyle='round,pad=0.005', linewidth=1.2,
    edgecolor='#d0cdc8', facecolor='none',
    transform=ax.transAxes, zorder=4))

plt.tight_layout(pad=0.3)
plt.savefig('results_table.png', dpi=180, bbox_inches='tight', facecolor=BG_TABLE)
plt.show()
print('✓ Saved: results_table.png')

## 💾 Section 12 — Save JSON & Download All Files

In [ ]:
output = {
    'experiment':   'Knowledge Structure and LLM Reasoning Quality — Real LLM',
    'timestamp':    datetime.now().isoformat(),
    'models':       list(MODELS.keys()),
    'domain':       'Synthetic Biomedical',
    'kb':           {'triples': len(TRIPLES), 'entities': len(ENTITIES)},
    'network':      {k: (round(v, 4) if isinstance(v, float) else v)
                     for k, v in metrics.items()
                     if k not in ('node_comm', 'degree_c', 'between_c')},
    'summary':      summary,
    'full_results': all_results,
}
with open('experiment_llm_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)
print('✓ Saved: experiment_llm_results.json')

# ── Auto-download all three files in Colab ────────────────────────────────────
try:
    from google.colab import files
    print('\nDownloading files...')
    files.download('experiment_llm_results.png')
    files.download('results_table.png')
    files.download('experiment_llm_results.json')
    print('✓ All files downloaded')
except ImportError:
    print('\nℹ Not running in Colab — files saved in current directory:')
    import os
    for f in ['experiment_llm_results.png', 'results_table.png', 'experiment_llm_results.json']:
        size = os.path.getsize(f) // 1024
        print(f'  {f}  ({size} KB)')

## 🔍 Section 13 — Inspect Full LLM Responses

Qualitative review — what the models actually said in each condition.

In [ ]:
for model_id, results in all_results.items():
    for r in results:
        for cond in ['plain', 'kg']:
            sc = r[cond]['scores']
            print(f'\n{"═"*65}')
            print(f'  [{model_id}]  [{r["question_id"]}]  [{cond.upper()}]')
            print(f'  kw={sc["keyword_recall"]:.0%}  '
                  f'hal={sc["hallucination_count"]}  '
                  f'depth={sc["reasoning_depth"]:.2f}  '
                  f'words={sc["response_words"]}')
            print(f'  Keywords found : {sc["keywords_found"]}')
            if sc['hallucinations']:
                print(f'  ⚠️  Hallucinations: {sc["hallucinations"]}')
            print(f'  {"─"*63}')
            resp = r[cond]['response']
            print(resp[:800] + ('...' if len(resp) > 800 else ''))

---

## ✅ Output Summary

| File | Description | Use |
|---|---|---|
| `experiment_llm_results.png` | 5-panel figure | Article figure, Substack image block |
| `results_table.png` | Styled results table | Substack / Medium table image |
| `experiment_llm_results.json` | Full results + LLM responses | GitHub repo, reproducibility |

---

**Article:** *From Data to Knowledge: Why Knowledge Graphs and Network Science Are a Likely Foundation of More Reliable AI*  
**GitHub:** `github.com/YOUR_USERNAME/knowledge-graph-llm-experiment`